# 바로핏 · 옆으로 누워 다리 들기
QEVD 원본 영상 5개를 자동 준비하고 사진 위 관절·각도·횟수·한국어 피드백을 표시합니다. **1번부터 순서대로 실행**하세요. API 키와 GPU는 필요 없습니다.

- 원본 운동명: `hip abductions (right leg)`.
- 비교: `average` 기준 후보 1개와 동작 변형/오류 후보 4개. `average`는 전문가가 모든 항목을 정자세로 보증한 라벨이 아닙니다.
- 출처 라벨과 자동 측정 평가는 별도로 표시합니다. 어두운 영상·가림·짧은 클립에서는 횟수가 0일 수 있습니다.
- 연구용 데이터: [QEVD 안내·이용 조건](https://www.qualcomm.com/developer/software/qevd-dataset), [공식 다운로드](https://www.qualcomm.com/developer/software/qevd-dataset/downloads).
- `side_lying_leg_raise`는 기존 Barofit v2에 대한 **확장 제안**입니다. 기존 3개 운동 전용 스키마에 그대로 통과하지 않습니다.


## 1. 설치
새 Colab 런타임에서 실행하세요. 설치 후 import 충돌이 있으면 런타임을 재시작하고 2번부터 실행합니다.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable,'-m','pip','install','-q','mediapipe==0.10.32','opencv-python-headless','numpy','pandas','matplotlib','pillow','jsonschema'])

## 2. 경로와 설정
기본값으로 공식 서버에서 필요한 5개 영상만 다운로드합니다. 전체 수십 GB 파일은 받지 않습니다. 저장 위치를 Drive로 바꾸려면 `SAVE_TO_DRIVE=True`로 바꾸세요. 기존 데이터 폴더 이름을 입력할 필요가 없습니다.

임계값은 개발용 가설입니다. 고관절의 실제 3D 각도나 의학적 정자세 기준이 아닙니다. 기본 카운트는 낮춤 → 올림 → 낮춤이 모두 관측될 때 1회 증가합니다.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, math, copy, hashlib, urllib.request, time, uuid, re
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, Image as ShowImage
MODE='official_samples'
SAVE_TO_DRIVE=False
SAMPLE_FPS=15
GIF_FPS=8
GIF_MAX_SECONDS=12
MAX_VIDEO_SECONDS=30
VISIBILITY=.45
TH=dict(lower=15.,raised=20.,target=35.,knee=155.,torso_horizontal=45.,stable=.08,warning_duration=.25,min_rep=.35,target_duration=1.2,max_gap=.3,ema_tau=.06)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
BASE_DIR=Path('/content/drive/MyDrive/Barofit/side_lying_leg_raise_results' if SAVE_TO_DRIVE else '/content/barofit_side_lying_leg_raise')
DATA_ROOT=BASE_DIR/'videos';DATA_ROOT.mkdir(parents=True,exist_ok=True)
CACHE_DIR=BASE_DIR/'_pose_cache';CACHE_DIR.mkdir(parents=True,exist_ok=True)
RUN_DIR=BASE_DIR/(datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:6]);RUN_DIR.mkdir(parents=True)
print('결과 저장:',RUN_DIR)


## 3. 원본 라벨 및 영상 5개 준비
ZIP의 필요한 바이트만 요청하고 CRC와 SHA-256으로 검증합니다. 서버가 Range를 지원하지 않거나 파일이 바뀌면 명확한 오류로 중단합니다. 전체 ZIP 다운로드로 전환하지 않습니다.

FitAQA의 같은 파일명과 원본 QEVD 영상 간 불일치 사례가 확인되어, 여기서는 **QEVD-FIT-COACH의 `fine_grained_labels.json`과 QEVD-FIT-300k 원본**을 직접 연결했습니다. 원본 라벨은 동작 변형의 이름이며 자동 평가의 정답으로 무조건 적용하지 않습니다.

In [ ]:
MANIFEST = [{'video_path': './00000828.mp4', 'labels': ['hip abductions (right leg) - average'], 'labels_descriptive': ['hip abductions (right leg) - User is doing average'], 'split': 'train', 'entry': {'name': 'QEVD-FIT-300k-Part-1/00000828.mp4', 'part': 1, 'disk': 0, 'offset': 220750772, 'csize': 368895, 'usize': 370964, 'crc': 4101297951, 'method': 8, 'flags': 0}, 'sha256': '971dfa8fe2ddbe18424caf904916c0339f58feb3cd13e39d37d0a054af1014c3', 'attribute': 'average'}, {'video_path': './00009154.mp4', 'labels': ['hip abductions (right leg) - low range of motion'], 'labels_descriptive': ['hip abductions (right leg) - User has low range of motion'], 'split': 'train', 'entry': {'name': 'QEVD-FIT-300k-Part-1/00009154.mp4', 'part': 1, 'disk': 0, 'offset': 2431458457, 'csize': 219379, 'usize': 221129, 'crc': 2007208406, 'method': 8, 'flags': 0}, 'sha256': 'ae3cb60e6f06e1f38d37b0f77717b858e117e37a470cfe646ea10dad4db2ba91', 'attribute': 'low range of motion'}, {'video_path': './00009410.mp4', 'labels': ['hip abductions (right leg) - leg not straight'], 'labels_descriptive': ["hip abductions (right leg) - User's legs are not straight"], 'split': 'train', 'entry': {'name': 'QEVD-FIT-300k-Part-1/00009410.mp4', 'part': 1, 'disk': 0, 'offset': 2500996972, 'csize': 250505, 'usize': 252359, 'crc': 2951064226, 'method': 8, 'flags': 0}, 'sha256': '4f0f1af45cd95c6e9be2236b66ef74d3bb7d5bb64494830a4458d71edeecac8c', 'attribute': 'leg not straight'}, {'video_path': './00007549.mp4', 'labels': ['hip abductions (right leg) - upper body not straight'], 'labels_descriptive': ["hip abductions (right leg) - User's upper body isn't straight"], 'split': 'train', 'entry': {'name': 'QEVD-FIT-300k-Part-1/00007549.mp4', 'part': 1, 'disk': 0, 'offset': 2002349585, 'csize': 149176, 'usize': 151572, 'crc': 1583319388, 'method': 8, 'flags': 0}, 'sha256': '7920ad4e9a492c8852061944933855357f73518e0fc3e88f4535fceae6688d52', 'attribute': 'upper body not straight'}, {'video_path': './00000205.mp4', 'labels': ['hip abductions (right leg) - as fast as possible'], 'labels_descriptive': ['hip abductions (right leg) - User is going as fast as possible'], 'split': 'train', 'entry': {'name': 'QEVD-FIT-300k-Part-1/00000205.mp4', 'part': 1, 'disk': 0, 'offset': 64018279, 'csize': 361823, 'usize': 363772, 'crc': 1528620343, 'method': 8, 'flags': 0}, 'sha256': '640b029552e0dd5f0f69f3ca36f9a7c29bd7261d57372ae0fb70400a85481d4d', 'attribute': 'as fast as possible'}]


In [ ]:
import urllib.request, urllib.error, struct, zlib, hashlib, json, time
from pathlib import Path
BASE='https://softwarecenter.qualcomm.com/api/download/software/dataset/AIDataset/Qualcomm_Exercise_Video_Dataset'

def url_for(part,disk):
    stem=f'QEVD-FIT-300k-Part-{part}'
    ext=('z01','z02','zip')[disk]
    return f'{BASE}/{stem}/{stem}.{ext}'

def fetch_range(url,start=None,end=None,suffix=None):
    requested=f'bytes=-{suffix}' if suffix else f'bytes={start}-{end}'
    error=None
    for attempt in range(3):
        try:
            req=urllib.request.Request(url,headers={'User-Agent':'BarofitResearch/1.0','Range':requested,'Accept-Encoding':'identity'})
            with urllib.request.urlopen(req,timeout=45) as r:
                if r.status!=206:raise RuntimeError('서버가 부분 다운로드를 지원하지 않습니다. 전체 파일은 받지 않습니다.')
                cr=r.headers.get('Content-Range','');span,total=cr.split(' ')[1].split('/');lo,hi=map(int,span.split('-'));total=int(total)
                if suffix:
                    if hi!=total-1 or lo!=max(0,total-suffix):raise RuntimeError('잘못된 Range 응답')
                elif lo!=start or hi!=min(end,total-1):raise RuntimeError('잘못된 Range 응답')
                n=hi-lo+1
                if n>32*1024**2:raise RuntimeError('부분 요청 한도 초과')
                data=r.read(n+1)
                if len(data)!=n:raise RuntimeError('불완전한 부분 다운로드')
                return data,total
        except Exception as e:
            error=e
            if attempt<2:time.sleep(1+attempt)
    raise RuntimeError(f'부분 다운로드 실패: {requested}: {error}')

def index_part(part,cache):
    cache=Path(cache);cache.mkdir(parents=True,exist_ok=True);p=cache/f'part{part}_index.json'
    if p.exists():return json.loads(p.read_text())
    tail,total=fetch_range(url_for(part,2),suffix=65557)
    i=tail.rfind(b'PK\x05\x06')
    if i<0:raise RuntimeError('ZIP 끝 레코드 없음')
    _,disk,cd_disk,n_disk,n_all,cd_size,cd_offset,comment=struct.unpack_from('<4s4H2LH',tail,i)
    if cd_disk!=2:raise RuntimeError('예상하지 못한 중앙 디렉터리 위치')
    if cd_size==0xffffffff or cd_offset==0xffffffff:raise RuntimeError('중앙 디렉터리 ZIP64 오프셋 어댑터 필요')
    raw,_=fetch_range(url_for(part,2),cd_offset,cd_offset+cd_size-1)
    out={};pos=0
    while pos+46<=len(raw) and raw[pos:pos+4]==b'PK\x01\x02':
        v=struct.unpack_from('<4s6H3L5H2L',raw,pos)
        flags,method,crc,csize,usize=v[3],v[4],v[7],v[8],v[9]
        nl,el,cl,ds,offset=v[10],v[11],v[12],v[13],v[16]
        name=raw[pos+46:pos+46+nl].decode('utf-8' if flags&2048 else 'cp437')
        extra=raw[pos+46+nl:pos+46+nl+el];ep=0
        while ep+4<=len(extra):
            tag,size=struct.unpack_from('<HH',extra,ep);val=extra[ep+4:ep+4+size];ep+=4+size
            if tag==1:
                off=0
                if usize==0xffffffff:usize=struct.unpack_from('<Q',val,off)[0];off+=8
                if csize==0xffffffff:csize=struct.unpack_from('<Q',val,off)[0];off+=8
                if offset==0xffffffff:offset=struct.unpack_from('<Q',val,off)[0];off+=8
                if ds==0xffff:ds=struct.unpack_from('<L',val,off)[0]
        out[Path(name).name]=dict(name=name,part=part,disk=ds,offset=offset,csize=csize,usize=usize,crc=crc,method=method,flags=flags)
        pos+=46+nl+el+cl
    if not out:raise RuntimeError('비어 있는 ZIP 인덱스')
    p.write_text(json.dumps(out));print('ZIP 인덱스:',part,len(out),flush=True);return out

def read_disks(part,disk,offset,n):
    chunks=[]
    while n:
        if disk>2:raise RuntimeError('분할 ZIP 범위를 벗어났습니다.')
        b,total=fetch_range(url_for(part,disk),offset,offset+min(n,16*1024**2)-1)
        chunks.append(b);n-=len(b);offset+=len(b)
        if offset==total:disk+=1;offset=0
    return b''.join(chunks)

def extract_entry(entry,dest,expected_sha):
    dest=Path(dest)
    if dest.exists() and expected_sha and hashlib.sha256(dest.read_bytes()).hexdigest()==expected_sha:return dest
    if entry['flags']&1:raise RuntimeError('암호화 ZIP은 지원하지 않습니다.')
    if entry['usize']>100*1024**2 or entry['csize']>100*1024**2:raise RuntimeError('단일 영상 100MB 한도 초과')
    head=read_disks(entry['part'],entry['disk'],entry['offset'],30)
    if head[:4]!=b'PK\x03\x04':raise RuntimeError('로컬 ZIP 헤더 불일치')
    nl,el=struct.unpack_from('<HH',head,26)
    # 전체 엔트리를 같은 시작점에서 읽어 분할 경계를 처리합니다.
    packed=read_disks(entry['part'],entry['disk'],entry['offset'],30+nl+el+entry['csize'])[30+nl+el:]
    if entry['method']==8:data=zlib.decompress(packed,-15)
    elif entry['method']==0:data=packed
    else:raise RuntimeError('지원하지 않는 ZIP 압축 방식')
    if len(data)!=entry['usize'] or (zlib.crc32(data)&0xffffffff)!=entry['crc']:raise RuntimeError('ZIP CRC 검증 실패')
    if expected_sha and hashlib.sha256(data).hexdigest()!=expected_sha:raise RuntimeError('파일 SHA-256 불일치')
    dest.parent.mkdir(parents=True,exist_ok=True);tmp=dest.with_suffix('.part');tmp.write_bytes(data);tmp.replace(dest)
    print('영상 검증 완료:',dest.name,round(len(data)/1e6,2),'MB',flush=True);return dest


labels_ko=['기준 후보: average','작은 가동 범위 후보','무릎 굽힘 후보','몸통 비정렬 후보','빠른 속도 비교 후보']
CLIPS=[]
for i,record in enumerate(MANIFEST):
    name=Path(record['video_path']).name
    path=extract_entry(record['entry'],DATA_ROOT/name,record['sha256'])
    CLIPS.append(dict(name=f'case_{i}',path=str(path),label=labels_ko[i],sourceLabel=record['labels'][0],side='right',start=0.,end=None))
(RUN_DIR/'source_manifest.json').write_text(json.dumps(MANIFEST,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.DataFrame(CLIPS)[['name','label','sourceLabel','path']])


## 4. 관절 추출
MediaPipe Pose Landmarker Full을 사용합니다. 최초 실행만 관절을 추출하고 이후에는 캐시를 재사용합니다. 양쪽 다리를 평균내지 않고, 원본 라벨에 명시된 **오른쪽 다리**를 평가합니다. 영상이 미러링되었거나 좌우 인식이 뒤바뀌었다면 미리보기 확인 후 `CLIPS[i]['side']='left'`로 바꾸고 6번부터 다시 실행하세요.

In [ ]:
FONT_PATH=Path('/content/barofit_fonts/NotoSansKR.ttf');FONT_PATH.parent.mkdir(parents=True,exist_ok=True)
def download(url,dest):
    dest=Path(dest);tmp=dest.with_suffix(dest.suffix+'.part')
    try:
        req=urllib.request.Request(url,headers={'User-Agent':'BarofitResearch/1.0'})
        with urllib.request.urlopen(req,timeout=60) as response,open(tmp,'wb') as f:
            while True:
                b=response.read(1024*1024)
                if not b:break
                f.write(b)
        tmp.replace(dest)
    finally:
        tmp.unlink(missing_ok=True)
if not FONT_PATH.exists():
    try:download('https://raw.githubusercontent.com/google/fonts/main/ofl/notosanskr/NotoSansKR%5Bwght%5D.ttf',FONT_PATH)
    except Exception as e:print('한국어 폰트 다운로드 실패:',type(e).__name__)

DATA={}
import mediapipe as mp
MODEL=Path('/content/models/pose_landmarker_full.task');MODEL.parent.mkdir(parents=True,exist_ok=True)
if not MODEL.exists():download('https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task',MODEL)
mh=hashlib.sha256(MODEL.read_bytes()).hexdigest()
opts=mp.tasks.vision.PoseLandmarkerOptions(base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL)),running_mode=mp.tasks.vision.RunningMode.VIDEO,num_poses=1)
for clip in CLIPS:
    path=Path(clip['path']);st=path.stat()
    key=hashlib.sha256(f'{path}|{st.st_size}|{st.st_mtime_ns}|{mh}|{SAMPLE_FPS}|{MAX_VIDEO_SECONDS}|{mp.__version__}'.encode()).hexdigest()
    cache=CACHE_DIR/(key+'.npz')
    if cache.exists():
        with np.load(cache,allow_pickle=False) as a: data={k:a[k] for k in a.files}
    else:
        cap=cv2.VideoCapture(str(path));fps=cap.get(cv2.CAP_PROP_FPS)
        if not cap.isOpened() or not math.isfinite(fps) or fps<=0:cap.release();raise ValueError(f'영상 FPS 확인 실패: {path}')
        total=cap.get(cv2.CAP_PROP_FRAME_COUNT)
        poses=[];times=[];dims=[];i=0;next_t=0.;last_ms=-1
        try:
            with mp.tasks.vision.PoseLandmarker.create_from_options(opts) as model:
                while True:
                    ok,bgr=cap.read()
                    if not ok:break
                    t=i/fps;i+=1
                    if t>MAX_VIDEO_SECONDS:break
                    if t+1e-8<next_t:continue
                    next_t=t+1/SAMPLE_FPS
                    h,w=bgr.shape[:2];scale=min(1,960/max(w,h))
                    rgb=cv2.cvtColor(cv2.resize(bgr,(round(w*scale),round(h*scale))),cv2.COLOR_BGR2RGB)
                    ms=max(last_ms+1,round(t*1000));last_ms=ms
                    result=model.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB,data=rgb),ms)
                    p=np.full((33,5),np.nan)
                    if result.pose_landmarks:p=np.array([[q.x,q.y,q.z,q.visibility,q.presence] for q in result.pose_landmarks[0]])
                    poses.append(p);times.append(t);dims.append([w,h])
                    if len(times)%100==0:print(path.name,round(t,1),'초',flush=True)
        finally:cap.release()
        if not poses:raise ValueError('관절 추출할 프레임이 없습니다.')
        if total>0 and i<min(total,MAX_VIDEO_SECONDS*fps)*.95:raise RuntimeError('영상 디코딩이 불완전합니다.')
        data=dict(poses=np.array(poses),times=np.array(times),dims=np.array(dims))
        np.savez_compressed(cache,**data)
    mask=(data['times']>=clip['start'])
    if clip['end'] is not None:mask&=data['times']<=clip['end']
    data={k:v[mask] for k,v in data.items()}
    if len(data['times'])<2:raise ValueError(f"구간이 너무 짧습니다: {clip['name']}")
    DATA[clip['name']]=data
print('관절 추출 완료:',len(DATA))

## 5. 측정·카운트·피드백 엔진
- 관측값: 영상 종횡비를 보정한 어깨–골반–무릎 각도, 무릎 각도.
- 올림각: `180 − 어깨·골반·무릎 각도`. 영상 평면에 투영된 값입니다.
- 카운트: 15° 이하 준비 → 20° 이상 → 15° 이하 복귀. 중간부터 시작한 첫 부분 동작, 인식이 끊긴 동작은 세지 않습니다.
- 회차 피드백: 최고 올림각 35° 미만, 무릎 각도 155° 미만 지속, 측정 구간 속도가 빠름을 실험적으로 평가합니다.
- 골반 회전 및 실제 척추 정렬은 `not_supported`입니다. 출처에 몸통 오류 라벨이 있어도 이 코드가 자동 판별했다고 표현하지 않습니다.
- `items/currentCodes`는 현재 관측, `lastRepetition`은 마지막 완료 회차입니다. 현재 안내와 회차 피드백을 별도 표시합니다.


In [ ]:
SIDES={'left':(11,23,25,27),'right':(12,24,26,28)}
MESSAGES={'range':'다리를 조금 더 올리세요.', 'knee_extension':'위쪽 무릎을 조금 더 펴세요.', 'pace':'반동 없이 조금 더 천천히 움직이세요.'}
def angle(a,b,c):
    u=np.asarray(a)-b;v=np.asarray(c)-b;den=np.linalg.norm(u)*np.linalg.norm(v)
    return float(np.degrees(np.arccos(np.clip(np.dot(u,v)/den,-1,1)))) if den>1e-8 else None

def measurements(pose,w,h):
    out={}
    for side,ids in SIDES.items():
        q=pose[list(ids)]
        if not np.isfinite(q).all() or np.min(q[:,3:5])<VISIBILITY:continue
        s,hip,knee,ank=q[:,:2]*[w,h];ha=angle(s,hip,knee);ka=angle(hip,knee,ank)
        if ha is None or ka is None:continue
        dx,dy=s-hip
        out[side]=dict(elevation=180-ha,knee=ka,hip=ha,torsoTilt=float(np.degrees(np.arctan2(abs(dy),abs(dx)))))
    return out

class SideRaiseEngine:
    version='side-lying-rules-0.1-experimental'
    def __init__(self,side='right'):
        self.side=side;self.count=0;self.last_t=None;self.last_rep=None;self.reps=[];self.finished=False;self.summary=None
        self.interrupt();self.last_lower=None
    def interrupt(self):
        self.phase='awaiting_lower';self.bound=None;self.candidate=None;self.active=None;self.filtered=None;self.last_lower=None
    def rep_feedback(self):
        if self.last_rep is None:return '아직 완료된 회차가 없습니다.'
        return ' '.join(MESSAGES[k] for k in self.last_rep['codes']) or '측정한 항목에서는 경고가 없습니다.'
    def update(self,t,raw):
        if not math.isfinite(t) or t<0:raise ValueError('시각은 0 이상 유한값이어야 합니다.')
        reason='finished' if self.finished else ('non_monotonic_timestamp' if self.last_t is not None and t<=self.last_t else None)
        base=dict(schemaVersion='2.0',type='frame.result',exercise='side_lying_leg_raise',engineVersion=self.version,timestampMs=t*1000,ignored=bool(reason),ignoredReason=reason,tracking='tracking_unavailable',side=self.side,count=self.count,holdSeconds=None,observedSeconds=None,measurementBasis=dict(count='completed_threshold_cycle',holdSeconds=None,observedSeconds=None),phase=self.phase,items={},codes=[],currentCodes=[],lastRepetition=None,metrics={},events=[],details={})
        if reason:return base
        dt=0 if self.last_t is None else t-self.last_t;self.last_t=t
        if dt>TH['max_gap']:
            self.interrupt();base['events'].append({'type':'interrupted','reason':'sample_gap'})
        v=raw.get(self.side)
        if v is None or not all(math.isfinite(v[k]) for k in ['elevation','knee','torsoTilt']) or v['torsoTilt']>TH['torso_horizontal']:
            self.interrupt();base.update(phase=self.phase,items={**{k:'tracking_unavailable' for k in MESSAGES},'pelvis_rotation':'not_supported','body_alignment':'not_supported'},lastRepetition=copy.deepcopy(self.last_rep),details=dict(guidance='전신과 위쪽 다리가 보이도록 촬영하세요.',repFeedback=self.rep_feedback(),source='rule_based'))
            return base
        alpha=1 if self.filtered is None or dt<=0 else 1-math.exp(-dt/TH['ema_tau'])
        old=self.filtered or v;self.filtered={k:old[k]+alpha*(v[k]-old[k]) for k in v}
        elev=self.filtered['elevation'];knee=self.filtered['knee']
        condition='lower' if elev<=TH['lower'] else ('raised' if elev>=TH['raised'] else None)
        if condition!=self.candidate:self.candidate=condition;self.bound=t
        stable=condition is not None and t-self.bound>=TH['stable']
        if self.phase=='awaiting_lower' and stable and condition=='lower':self.phase='ready';self.last_lower=t
        if self.phase=='ready':
            if condition=='lower':self.last_lower=t
            if stable and condition=='raised':
                self.phase='raised';self.active=dict(start=self.last_lower if self.last_lower is not None else t,peak=elev,kneeRun=0.,kneeFlag=False)
        a=self.active
        if a is not None:
            a['peak']=max(a['peak'],elev);a['kneeRun']=a['kneeRun']+dt if knee<TH['knee'] else 0.
            a['kneeFlag']|=a['kneeRun']>=TH['warning_duration']
        items={'range':'pending_feedback','knee_extension':('warning' if a and a['kneeRun']>=TH['warning_duration'] else 'within_reference') if a else 'pending_feedback','pace':'pending_feedback','pelvis_rotation':'not_supported','body_alignment':'not_supported'}
        current=[k for k,v in items.items() if v=='warning']
        if self.phase=='raised' and stable and condition=='lower':
            duration=t-a['start']
            if duration>=TH['min_rep']:
                self.count+=1
                states=dict(range='within_reference' if a['peak']>=TH['target'] else 'warning',knee_extension='warning' if a['kneeFlag'] else 'within_reference',pace='warning' if duration<TH['target_duration'] else 'within_reference',pelvis_rotation='not_supported',body_alignment='not_supported')
                self.last_rep=dict(number=self.count,timestampMs=t*1000,items=states,codes=[k for k,v in states.items() if v=='warning'],metrics=dict(peakElevationDeg=a['peak'],thresholdCycleSeconds=duration,side=self.side))
                self.reps.append(copy.deepcopy(self.last_rep));base['events'].append({'type':'repetition.completed','number':self.count})
            else:base['events'].append({'type':'attempt.discarded','reason':'too_short'})
            self.phase='ready';self.active=None;self.last_lower=t
        guidance='다리를 천천히 올리세요.' if self.phase=='ready' else ('반동 없이 천천히 내리세요.' if self.phase=='raised' else '다리를 낮춘 준비 자세부터 시작하세요.')
        if current:guidance=' '.join(MESSAGES[k] for k in current)
        base.update(tracking='tracking',count=self.count,phase=self.phase,items=items,currentCodes=current,codes=list(dict.fromkeys(current+(self.last_rep['codes'] if self.last_rep else []))),lastRepetition=copy.deepcopy(self.last_rep),metrics=dict(elevationDeg=elev,kneeDeg=knee,sides=raw),details=dict(guidance=guidance,repFeedback=self.rep_feedback(),source='rule_based',experimental=True))
        return base
    def finish(self):
        if self.summary is not None:return copy.deepcopy(self.summary)
        self.finished=True;stats={}
        for key in [*MESSAGES,'pelvis_rotation','body_alignment']:
            states=[r['items'][key] for r in self.reps];c=states.count('within_reference');w=states.count('warning');u=states.count('unknown');na=states.count('not_supported')
            stats[key]=dict(aggregationBasis='repetition',unit='repetitions',compliant=c,warning=w,unknown=u,notApplicable=na,evaluated=c+w,compliancePct=100*c/(c+w) if c+w else None)
        self.summary=dict(schemaVersion='2.0',type='set.completed',exercise='side_lying_leg_raise',engineVersion=self.version,timestampMs=None if self.last_t is None else self.last_t*1000,summary=dict(schemaVersion='2.0',exercise='side_lying_leg_raise',engineVersion=self.version,count=self.count,holdSeconds=None,observedSeconds=None,gapSeconds=None,items=stats,details=dict(incompleteAttempt=self.active is not None,thresholds=TH,experimental=True)))
        return copy.deepcopy(self.summary)


## 6. 반환 스키마 검증과 분석
각 프레임 JSONL, 세트 요약 JSON, 확장 스키마를 저장합니다. 출처 라벨은 `details.sourceLabel`로 보존합니다.

In [ ]:
from jsonschema import Draft202012Validator
STATUS=['within_reference','warning','tracking_unavailable','pending_feedback','body_unavailable','body_deviation','interrupted','unknown','not_applicable','disabled','not_supported']
def obj(properties,required=None):
    return dict(type='object',properties=properties,required=list(properties) if required is None else required,additionalProperties=False)
def nullable(kind,minimum=0):
    return dict(type=[kind,'null'],minimum=minimum)
ITEMS=dict(type='object',additionalProperties=dict(enum=STATUS))
METRICS=dict(type='object')
CODES=dict(type='array',items=dict(type='string'))
REP=obj(dict(number=nullable('integer',1),timestampMs=nullable('number'),items=ITEMS,codes=CODES,metrics=METRICS))
BASIS=dict(count='completed_threshold_cycle',holdSeconds=None,observedSeconds=None)
FRAME_SCHEMA=obj(dict(schemaVersion=dict(const='2.0'),type=dict(const='frame.result'),exercise=dict(enum=['side_lying_leg_raise']),engineVersion=dict(type='string'),timestampMs=dict(type='number',minimum=0),ignored=dict(type='boolean'),ignoredReason=dict(enum=[None,'finished','non_monotonic_timestamp']),tracking=dict(enum=['tracking','selecting_side','tracking_unavailable']),side=dict(enum=[None,'left','right']),count=dict(type='integer',minimum=0),holdSeconds=dict(type='null'),observedSeconds=dict(type='null'),measurementBasis=dict(const=BASIS),phase=dict(type=['string','null']),items=ITEMS,codes=CODES,currentCodes=CODES,lastRepetition=dict(oneOf=[dict(type='null'),REP]),metrics=METRICS,events=dict(type='array',items=dict(type='object',required=['type'],properties=dict(type=dict(type='string')))),details=METRICS))
FRAME_SCHEMA['$schema']='https://json-schema.org/draft/2020-12/schema'
FRAME_SCHEMA['title']='Proposed Barofit v2 extension: side_lying_leg_raise frames'
FRAME_SCHEMA['allOf']=[
    {'if':{'properties':{'ignored':{'const':True}}},'then':{'properties':{'ignoredReason':{'enum':['finished','non_monotonic_timestamp']},'items':{'maxProperties':0},'codes':{'maxItems':0},'currentCodes':{'maxItems':0},'lastRepetition':{'type':'null'},'events':{'maxItems':0}}},'else':{'properties':{'ignoredReason':{'type':'null'}}}}
]
AGG=obj(dict(aggregationBasis=dict(const='repetition'),unit=dict(const='repetitions'),compliant=dict(type='integer',minimum=0),warning=dict(type='integer',minimum=0),unknown=dict(type='integer',minimum=0),notApplicable=dict(type='integer',minimum=0),evaluated=dict(type='integer',minimum=0),compliancePct=dict(type=['number','null'],minimum=0,maximum=100)))
AGG['allOf']=[{'if':{'properties':{'evaluated':{'const':0}}},'then':{'properties':{'compliancePct':{'type':'null'}}},'else':{'properties':{'compliancePct':{'type':'number'}}}}]
SUMMARY=obj(dict(schemaVersion=dict(const='2.0'),exercise=dict(enum=['side_lying_leg_raise']),engineVersion=dict(type='string'),count=dict(type='integer',minimum=0),holdSeconds=dict(type='null'),observedSeconds=dict(type='null'),gapSeconds=dict(type='null'),items=dict(type='object',additionalProperties=AGG),details=METRICS))
SET_SCHEMA=obj(dict(schemaVersion=dict(const='2.0'),type=dict(const='set.completed'),exercise=dict(enum=['side_lying_leg_raise']),engineVersion=dict(type='string'),timestampMs=nullable('number'),summary=SUMMARY))
SET_SCHEMA['$schema']=FRAME_SCHEMA['$schema']
SET_SCHEMA['title']='Proposed Barofit v2 extension: side_lying_leg_raise set summary'
SET_SCHEMA['oneOf']=[dict(properties=dict(exercise=dict(const=e),summary=dict(properties=dict(exercise=dict(const=e))))) for e in ['side_lying_leg_raise']]
Draft202012Validator.check_schema(FRAME_SCHEMA);Draft202012Validator.check_schema(SET_SCHEMA)
FRAME_VALIDATOR=Draft202012Validator(FRAME_SCHEMA)
SET_VALIDATOR=Draft202012Validator(SET_SCHEMA)
def validate_result(result):
    json.dumps(result,allow_nan=False)
    if result['type']=='frame.result':FRAME_VALIDATOR.validate(result)
    else:
        SET_VALIDATOR.validate(result)
        assert result['exercise']==result['summary']['exercise']
        assert result['engineVersion']==result['summary']['engineVersion']
        for item in result['summary']['items'].values():
            assert item['evaluated']==item['compliant']+item['warning']
            expected=100*item['compliant']/item['evaluated'] if item['evaluated'] else None
            assert item['compliancePct']==expected
            assert sum(item[k] for k in ['compliant','warning','unknown','notApplicable'])==result['summary']['count']
for name,schema in [('frame_result_extension.schema.json',FRAME_SCHEMA),('set_completed_extension.schema.json',SET_SCHEMA)]:
    (RUN_DIR/name).write_text(json.dumps(schema,ensure_ascii=False,indent=2),encoding='utf-8')
print('확장 스키마 생성 완료. 기존 pushup/plank/bird_dog 전용 스키마는 수정하지 않습니다.')

In [ ]:
for clip in CLIPS:
    d=DATA[clip['name']];engine=SideRaiseEngine(clip['side']);results=[]
    for t,p,(w,h) in zip(d['times'],d['poses'],d['dims']):
        r=engine.update(float(t),measurements(p,w,h));r['details']['sourceLabel']=clip['sourceLabel'];validate_result(r);results.append(r)
    completed=engine.finish();validate_result(completed)
    d.update(results=results,completed=completed)
    (RUN_DIR/(clip['name']+'_frames.jsonl')).write_text('\n'.join(json.dumps(r,ensure_ascii=False,allow_nan=False) for r in results),encoding='utf-8')
    (RUN_DIR/(clip['name']+'_set.json')).write_text(json.dumps(completed,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
    tracking=sum(r['tracking']=='tracking' for r in results)/len(results)
    print(clip['label'],'count=',engine.count,'tracking=',round(100*tracking,1),'%',engine.rep_feedback())
    if tracking<.5:print('인식 가능한 프레임이 적습니다. 영상을 바꾸거나 촬영 각도를 확인하세요. 강제로 횟수나 경고를 생성하지 않습니다.')
(RUN_DIR/'run_config.json').write_text(json.dumps(dict(mode=MODE,thresholds=TH,clips=CLIPS,sampleFPS=SAMPLE_FPS,experimental=True),ensure_ascii=False,indent=2),encoding='utf-8')

## 7. 원본 사진 위 스켈레톤·각도·피드백 비교 GIF 4개
왼쪽은 기준 후보, 오른쪽은 비교 후보입니다. 각 영상 시작 시각을 맞추며 동작 위상을 강제로 맞추지 않습니다. GIF가 다시 재생되면 화면 숫자도 처음으로 돌아갑니다. 이는 분석 카운트를 재설정한 것이 아닙니다.

초록색은 왼쪽, 파란색은 오른쪽 관절입니다. 횟수와 피드백은 선택한 다리만 평가합니다. 영상 길이가 달라 공통 재생 구간 이후의 횟수는 GIF에 안 보일 수 있으므로 JSON의 최종 횟수도 확인하세요.

In [ ]:
def font(size):
    return ImageFont.truetype(str(FONT_PATH),size) if FONT_PATH.exists() else ImageFont.load_default()
def base_frame(clip,t):
    cap=cv2.VideoCapture(clip['path'])
    try:
        cap.set(cv2.CAP_PROP_POS_MSEC,float(t)*1000);ok,bgr=cap.read()
        if not ok:return Image.new('RGB',(640,360),'#222222')
        return Image.fromarray(cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB))
    finally:cap.release()
def panel(clip,j):
    d=DATA[clip['name']];p=d['poses'][j];r=d['results'][j]
    im=base_frame(clip,d['times'][j]);im.thumbnail((560,400))
    canvas=Image.new('RGB',(560,590),'#101923');x0=(560-im.width)//2;canvas.paste(im,(x0,45));draw=ImageDraw.Draw(canvas)
    def point(k):return (x0+float(p[k,0])*im.width,45+float(p[k,1])*im.height)
    for side,ids in SIDES.items():
        col='#51e0a1' if side=='left' else '#68b8ff'
        for a,b in zip(ids[:-1],ids[1:]):
            if np.isfinite(p[[a,b]]).all() and np.min(p[[a,b],3:])>=VISIBILITY:draw.line([point(a),point(b)],fill=col,width=4)
        for k in ids:
            if np.isfinite(p[k]).all() and min(p[k,3:])>=VISIBILITY:
                x,y=point(k);draw.ellipse((x-4,y-4,x+4,y+4),fill=col)
    title=clip['label'] if FONT_PATH.exists() else clip['name']
    draw.text((12,8),title,fill='white',font=font(21))
    draw.text((12,455),f"Count: {r['count']} | t: {d['times'][j]:.1f}s | {r['tracking']}",fill='white',font=font(18))
    ev=r['metrics'].get('elevationDeg');kn=r['metrics'].get('kneeDeg')
    draw.text((12,482),f"Elevation: {ev:.1f}  Knee: {kn:.1f}" if ev is not None else 'Angles: unavailable',fill='#b1cbe0',font=font(18))
    guidance=r['details'].get('guidance','') if FONT_PATH.exists() else ', '.join(r['currentCodes']) or r['phase']
    last=r['details'].get('repFeedback','') if FONT_PATH.exists() else ('Last rep: '+','.join(r['lastRepetition']['codes']) if r['lastRepetition'] else 'No completed rep')
    for k,txt in enumerate([guidance,last]):
        for line in range(2):draw.text((12,510+k*34+line*16),txt[line*31:(line+1)*31],fill='#ffd479' if k==0 else '#c6d6e4',font=font(14))

    for side,ids in SIDES.items():
        values=measurements(p,im.width,im.height).get(side)
        if values:
            x,y=point(ids[2]);draw.text((x,y),f"{values['knee']:.0f} deg",font=font(14),fill='yellow')
    draw.text((12,426),'SOURCE: '+clip['sourceLabel'].split(' - ')[-1],font=font(15),fill='#b1cbe0')
    return canvas
ref=CLIPS[0]
for other in CLIPS[1:]:
    dr=DATA[ref['name']];do=DATA[other['name']]
    duration=min(GIF_MAX_SECONDS,dr['times'][-1]-dr['times'][0],do['times'][-1]-do['times'][0]);frames=[]
    for t in np.arange(0,duration,1/GIF_FPS):
        a=int(np.searchsorted(dr['times'],dr['times'][0]+t,side='right')-1)
        b=int(np.searchsorted(do['times'],do['times'][0]+t,side='right')-1)
        canvas=Image.new('RGB',(1120,590));canvas.paste(panel(ref,a),(0,0));canvas.paste(panel(other,b),(560,0));frames.append(canvas)
    if frames:
        dest=RUN_DIR/f"compare_{ref['name']}_{other['name']}.gif"
        frames[0].save(dest,save_all=True,append_images=frames[1:],duration=round(1000/GIF_FPS),loop=0,optimize=False)
        print(dest.name);display(ShowImage(filename=str(dest)))

## 8. 합쳐진 그래프와 개별 6개 각도 그래프
양쪽 각도는 관찰용으로 보존합니다. 카운트·평가는 선택 다리를 사용합니다. 각도 그래프의 빈 부분은 인식 불가이며 0도나 정자세가 아닙니다.

In [ ]:
rows=[]
fig,ax=plt.subplots(2,1,figsize=(11,7),constrained_layout=True)
for clip in CLIPS:
    d=DATA[clip['name']];rs=d['results'];ts=d['times']-d['times'][0]
    elev=[r['metrics'].get('elevationDeg',np.nan) for r in rs]
    ax[0].plot(ts,elev,label=clip['name']);ax[1].step(ts,[r['count'] for r in rs],where='post',label=clip['name'])
    fig2,axs=plt.subplots(3,2,figsize=(11,8),constrained_layout=True)
    table={'timeSeconds':d['times'],'count':[r['count'] for r in rs]}
    for col,side in enumerate(SIDES):
        for row,key in enumerate(['hip','knee','elevation']):
            vals=[r['metrics'].get('sides',{}).get(side,{}).get(key,np.nan) for r in rs]
            table[side+'_'+key+'Deg']=vals;axs[row,col].plot(ts,vals);axs[row,col].set_title(side+' '+key);axs[row,col].set_ylabel('degrees');axs[row,col].set_xlabel('seconds')
    fig2.savefig(RUN_DIR/(clip['name']+'_six_angles.png'),dpi=130);plt.close(fig2)
    pd.DataFrame(table).to_csv(RUN_DIR/(clip['name']+'_measurements.csv'),index=False)
    summary=d['completed']['summary']
    for key,st in summary['items'].items():rows.append(dict(case=clip['name'],label=clip['label'],item=key,**st))
ax[0].axhline(TH['target'],color='gray',ls='--',label='experimental target');ax[0].set_ylabel('elevation (degrees)')
ax[1].set_ylabel('completed cycles');ax[1].set_xlabel('relative seconds')
for a in ax:a.legend();a.grid(alpha=.2)
fig.savefig(RUN_DIR/'combined.png',dpi=140);plt.show()
report=pd.DataFrame(rows);report.to_csv(RUN_DIR/'summary.csv',index=False);display(report)
print('결과 폴더:',RUN_DIR)

## 9. 카운트·예외·스키마 자동 점검
합성 측정값을 이용한 엔진 점검이며 실제 영상 정확도 평가와 다릅니다.

In [ ]:
def test_stream(missing=False):
    e=SideRaiseEngine();frames=[]
    for t in np.arange(0,7,.05):
        u=t%3
        elev=5 if u<.5 else (5+50*(u-.5) if u<1.5 else (55-50*(u-1.5) if u<2.5 else 5))
        raw={'right':dict(elevation=elev,knee=175.,hip=180-elev,torsoTilt=0.)}
        if missing and 1.4<t%3<2.5:raw={}
        r=e.update(float(t),raw);validate_result(r);frames.append(r)
    return e,frames
engine,frames=test_stream();assert engine.count==2
r=engine.update(engine.last_t,{});assert r['ignoredReason']=='non_monotonic_timestamp';validate_result(r)
summary=engine.finish();validate_result(summary)
r=engine.update(10,{});assert r['ignoredReason']=='finished';validate_result(r)
engine,frames=test_stream(True);assert engine.count==0
engine=SideRaiseEngine();assert engine.finish()['summary']['items']['range']['compliancePct'] is None
print('완료 2회, 가림 후 미완료 제외, 중복 시각, 종료 후 입력, 집계 점검 PASS')


## 10. 결과 묶음 다운로드
원본 영상은 묶음에 포함하지 않습니다. 결과에는 원본 영상으로 만든 GIF가 포함되므로 원본 데이터의 연구용 이용 조건을 따르세요.

In [ ]:
import shutil
archive=shutil.make_archive(str(RUN_DIR),'zip',RUN_DIR)
print('결과:',RUN_DIR,'\nZIP:',archive)
from IPython.display import FileLink
display(FileLink(archive))
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass
